[[]]

In [2]:
import random
from termcolor import cprint, colored

class Board:
    WIDTH = 4
    HEIGHT = 4
    
    def __init__(self):
        self.matrix = [["O" for _ in range(self.HEIGHT)] for _ in range(self.WIDTH)]
        self.valid_coords = self.gen_valid_coords()
        
    def _repr_html_(self):
        color_map = {
            "W": (50,50,50),
            "P": (25,25,25),
            "B": (50,25,25),
            "G": (25,25,50),
            "O": (100,100,100),
            "S": (25,50,25),
            "X": (10,10,10)
        }
        
        html = ""
        for row in self.matrix:
            for ftypes in row:
                r,g,b = (0,0,0)
                for ftype in ftypes:
                    ra, ga, ba = color_map[ftype]
                    color = (r + ra, g + ga, b + ba)
                
                
                html += (
                    f'<span style="'
                    f'color: rgb(255,255,255);'
                    f'background-color: rgb{color};'
                    f'padding:2px;'
                    f'font-family:monospace;'
                    f'width: 30px;'
                    f'height: 30px;'
                    f'display: block;'
                    f'float: left;'
                    f'">'
                    f'{ftypes}'
                    f'</span>'
                )

            html += "<br>"

        return html

    
    def initialize(self):
        wumpus = Feature("W", Feature.CARDINAL, "S", -1000)
        pit1 = Feature("P", Feature.CARDINAL, "B", -1000)
        pit2 = Feature("P", Feature.CARDINAL, "B", -1000)
        pit3 = Feature("P", Feature.CARDINAL, "B", -1000)
        gold = Feature("G", Feature.LOCAL, "X", 1000)
        
        features = [pit1, pit2, pit3, gold, wumpus]
        
        
        for feature in features:
            idx = random.choice(range(len(self.valid_coords)))
            x, y = self.valid_coords[idx]
            
            if self.matrix[x][y] == "O":
                self.matrix[x][y] = ""
                
            self.place_feature(feature, x, y, idx)
            self.change_env(feature, x, y)

    
    def gen_valid_coords(self):
        # Define your bounds
        rows = range(self.WIDTH)
        cols = range(self.HEIGHT)

        # Define excluded coordinates as a set for O(1) lookup
        excluded = {(0, 0), (0, 1), (1, 0)}

        # Generate all possible coordinates excluding the forbidden ones
        return [
            (r, c) 
            for r in rows 
            for c in cols 
            if (r, c) not in excluded
        ]
    
    def place_feature(self, feature, x, y, idx):
        if feature.ftype == "P":
            self.valid_coords.pop(idx) 
    
        self.matrix[x][y] += feature.ftype


        
    def colored_matrix(self):
        pass
            
    
    def change_env(self, feature, x, y):
        for direction in feature.impact:
            dx, dy = direction
            
            if 0 <= x+dx <= 3 and 0 <= y+dy <= 3:
                if self.matrix[x+dx][y+dy] == "O":
                      self.matrix[x+dx][y+dy] = ""
                        
                self.matrix[x+dx][y+dy] += feature.impact_type
            else:
                continue
        

class Feature:
    CARDINAL = [(0,1),(1,0),(-1,0),(0,-1)]
    LOCAL = [(0,0)]
    
    def __init__(self, ftype, impact, impact_type, scoring):
        self.ftype = ftype
        self.impact = impact
        self.impact_type = impact_type
        self.scoring = scoring
        
    def __repr__(self):
        return self.ftype
    





In [3]:
class ProceduralAgent:
    def __init__(self, board_width=4, board_height=4):
        self.width = board_width
        self.height = board_height
        
        # Procedural agents have to manually track every specific type of state
        self.visited = set()
        self.safe_squares = set()
        
        # Tracking hazards requires explicit lists of "suspects"
        self.suspected_pits = set()
        self.suspected_wumpus = set()
        
        # We manually declare the starting square as safe
        self.safe_squares.add((0, 0))

    def perceive_and_update(self, x, y, board_matrix):
        """Hardcoded logic to interpret sensors and update sets."""
        self.visited.add((x, y))
        self.safe_squares.add((x, y)) # If we are alive here, it's safe
        
        cell_contents = board_matrix[x][y]
        neighbors = self._get_neighbors(x, y)
        
        # --- PROCEDURAL RULE 1: BREEZE LOGIC ---
        if "B" in cell_contents:
            # We feel a breeze. If neighbors aren't already known to be safe, suspect them.
            for nx, ny in neighbors:
                if (nx, ny) not in self.safe_squares:
                    self.suspected_pits.add((nx, ny))
        else:
            # No breeze! We must manually write the code to clear suspects.
            for nx, ny in neighbors:
                self.safe_squares.add((nx, ny))
                if (nx, ny) in self.suspected_pits:
                    self.suspected_pits.remove((nx, ny)) # Procedural elimination
                    
        # --- PROCEDURAL RULE 2: STENCH LOGIC ---
        if "S" in cell_contents:
            for nx, ny in neighbors:
                if (nx, ny) not in self.safe_squares:
                    self.suspected_wumpus.add((nx, ny))
        else:
            # No stench! Manually clear wumpus suspects.
            for nx, ny in neighbors:
                self.safe_squares.add((nx, ny))
                if (nx, ny) in self.suspected_wumpus:
                    self.suspected_wumpus.remove((nx, ny))

    def check_if_safe(self, x, y):
        """Checks if our hardcoded lists guarantee safety."""
        # It's only safe if it's explicitly in our safe_squares list
        # OR if it's completely absent from our suspect lists
        if (x, y) in self.safe_squares:
            return True
        if (x, y) not in self.suspected_pits and (x, y) not in self.suspected_wumpus:
            return True
        return False

    def _get_neighbors(self, x, y):
        """Helper to find valid grid neighbors (0-indexed)."""
        neighbors = []
        for dx, dy in [(0, 1), (1, 0), (-1, 0), (0, -1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < self.width and 0 <= ny < self.height:
                neighbors.append((nx, ny))
        return neighbors

In [4]:
board = Board()
board.matrix
board.initialize()
board.matrix

board